## What is UDP?



UDP is a way for two devices to send short messages to each other over a network.

Think of it like throwing a postcard
 -You write a short message
You write the destination address
You throw it into the network
You don’t wait for a reply
You don’t know if it arrives

But it is very fast

That is UDP.

# UDP Listener (Simple)

We create the UDP socket once and bind it to a port.
Then we use another cell to read from it.

In [1]:
import socket

UDP_IP = "0.0.0.0"
UDP_PORT = 8050   # <-- change to a free port, and use same on Arduino

# If a socket already exists in this notebook, close it first
try:
    sock.close()
except NameError:
    pass
except OSError:
    pass

sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.bind((UDP_IP, UDP_PORT))

print(f"Listening for UDP on {UDP_IP}:{UDP_PORT}")


Listening for UDP on 0.0.0.0:8050


In [2]:
import pandas as pd
from datetime import datetime

df = pd.DataFrame(columns=["timestamp", "sender", "value"])
df


,timestamp,sender,value


## Read one packet + log to pandas

In [3]:
print("Waiting for 1 UDP message...")

data, addr = sock.recvfrom(4096)          # this blocks until a packet arrives
text = data.decode(errors="ignore").strip()
sender = f"{addr[0]}:{addr[1]}"
timestamp = datetime.now().isoformat()

print("Received:", text, "from", sender)

df.loc[len(df)] = [timestamp, sender, text]
df.tail()


Waiting for 1 UDP message...
Received: -0.03 from 192.168.1.238:8000


,timestamp,sender,value
0,2025-12-03T21:57:56.200598,192.168.1.238:8000,-0.03


In [ ]:
df.to_csv("udp_data.csv", index=False)
print("Saved to udp_data.csv")


In [39]:
sock.close()
print("Socket closed")


Socket closed


---

# Continuous UDP listener

- runs in the background
- keeps listening and logging
- stores everything in a pandas DataFrame

## Create the DataFrame 

In [ ]:
import pandas as pd
from datetime import datetime

df = pd.DataFrame(columns=["timestamp", "sender", "value"])
df

## Start a continuous UDP listener in the background

In [4]:
import socket
import threading

UDP_IP = "0.0.0.0"
UDP_PORT = 8050  # choose a port and use the same on Arduino / sender

print(f"Starting UDP listener on {UDP_IP}:{UDP_PORT} ...")

# If an old socket exists, try to close it
try:
    sock.close()
except NameError:
    pass
except OSError:
    pass

# Create and bind socket
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
sock.bind((UDP_IP, UDP_PORT))

running = True  # flag to control the listener thread

def udp_listener():
    global df, running
    print("UDP listener thread is running...")

    while running:
        try:
            data, addr = sock.recvfrom(4096)  # blocking: waits for data
            text = data.decode(errors="ignore").strip()
            sender = f"{addr[0]}:{addr[1]}"
            timestamp = datetime.now().isoformat()

            print(f"[{timestamp}] {sender} -> {text}")

            # append to DataFrame
            new_row = {
                "timestamp": timestamp,
                "sender": sender,
                "value": text
            }
            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

        except OSError:
            # socket closed → exit loop
            break
        except Exception as e:
            print("Error receiving UDP:", e)
            break

listener_thread = threading.Thread(target=udp_listener, daemon=True)
listener_thread.start()


Starting UDP listener on 0.0.0.0:8050 ...
UDP listener thread is running...


[2025-12-03T21:58:16.625169] 192.168.1.238:8000 -> 0.34
[2025-12-03T21:58:16.811935] 192.168.1.238:8000 -> 0.30
[2025-12-03T21:58:17.006719] 192.168.1.238:8000 -> 0.29
[2025-12-03T21:58:17.195285] 192.168.1.238:8000 -> 0.20
[2025-12-03T21:58:17.388594] 192.168.1.238:8000 -> 0.27
[2025-12-03T21:58:17.577906] 192.168.1.238:8000 -> 0.29
[2025-12-03T21:58:17.764965] 192.168.1.238:8000 -> 0.21
[2025-12-03T21:58:17.950493] 192.168.1.238:8000 -> 0.30
[2025-12-03T21:58:18.132813] 192.168.1.238:8000 -> 0.35
[2025-12-03T21:58:18.329459] 192.168.1.238:8000 -> 0.28
[2025-12-03T21:58:18.509595] 192.168.1.238:8000 -> 0.31
[2025-12-03T21:58:18.698817] 192.168.1.238:8000 -> 0.37
[2025-12-03T21:58:18.890658] 192.168.1.238:8000 -> 0.38
[2025-12-03T21:58:19.072832] 192.168.1.238:8000 -> 0.32
[2025-12-03T21:58:19.265722] 192.168.1.238:8000 -> 0.32
[2025-12-03T21:58:19.458756] 192.168.1.238:8000 -> 0.23
[2025-12-03T21:58:19.650025] 192.168.1.238:8000 -> 0.31
[2025-12-03T21:58:19.839851] 192.168.1.238:8000 

After running this cell:

- The socket is bound once
- A background thread is started
- Every UDP packet sent to your_PC_IP : 8050 is printed and logged into df

## Check what has been received so far

In [5]:
df.tail()


,timestamp,sender,value
79,2025-12-03T21:58:31.321379,192.168.1.238:8000,-0.07
80,2025-12-03T21:58:31.511055,192.168.1.238:8000,0.02
81,2025-12-03T21:58:31.695528,192.168.1.238:8000,0.00
82,2025-12-03T21:58:31.882084,192.168.1.238:8000,0.02
83,2025-12-03T21:58:32.069895,192.168.1.238:8000,0.08


## Save everything to CSV

In [46]:
df.to_csv("udp_data.csv", index=False)
print("Saved to udp_data.csv")


Saved to udp_data.csv


## Stop continuous listening

When you’re done listening and logging:

In [6]:
running = False      # tell the thread to stop
sock.close()         # unblock recvfrom and close the socket
print("UDP listener stopped and socket closed.")


UDP listener stopped and socket closed.
